# 🤖 AI Data Mapping Agent — with Google Gemini LLM

> **PRD Summary:** An AI-powered agent that ingests Excel/CSV files, automatically maps columns to a  
> fixed target schema via Google Gemini 2.5 Flash/Pro, handles transformations (melt, date combining),  
> provides a human-in-the-loop UI (Streamlit), and exports cleaned data + logs to a multi-sheet Excel.

---
**Architecture Overview**

```
CSV / Excel  →  Ingest  →  Detect Patterns  →  Gemini LLM Mapping  →  Human Review
                                                                              ↓
                                                              Transformations (melt / date)
                                                                              ↓
                                                              Type Validation (Pydantic)
                                                                              ↓
                                                              Export Excel (Results + Logs)
```


## Section 1 — Install & Import Dependencies

In [21]:
# Install dependencies (run once)
import subprocess, sys
pkgs = [
    "pandas", "openpyxl", "xlsxwriter",
    "google-genai", "pydantic",
    "streamlit", "ipywidgets", "pytest", "python-dotenv", "Faker",
]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("✅ All packages installed.")


✅ All packages installed.


In [22]:
from __future__ import annotations

import io, json, os, re, sys, random, subprocess
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, field_validator

# Add project root to path so agent package is importable from this notebook
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Optional: load .env for GEMINI_API_KEY
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
print(f"GEMINI_API_KEY set: {'✅ Yes' if GEMINI_API_KEY else '❌ No — heuristic-only mode'}")


GEMINI_API_KEY set: ✅ Yes


## Section 2 — Define Target Schema

In [23]:
# Fixed target schema — edit this dict to evolve the schema
TARGET_SCHEMA: dict[str, dict[str, Any]] = {
    "Date":          {"dtype": "date",  "required": True,  "description": "Calendar date (YYYY-MM-DD)",
                      "aliases": ["date","day","report_date","transaction_date","period","week","month"]},
    "Campaign_Name": {"dtype": "str",   "required": True,  "description": "Marketing campaign name",
                      "aliases": ["campaign","campaign_name","campaign_id","ad_campaign","campaign name"]},
    "Channel":       {"dtype": "str",   "required": True,  "description": "Marketing channel (Google, Facebook, TV…)",
                      "aliases": ["channel","platform","media","source","network","medium"]},
    "Region":        {"dtype": "str",   "required": False, "description": "Geographic region",
                      "aliases": ["region","geo","country","market","territory","location"]},
    "Product":       {"dtype": "str",   "required": False, "description": "Product or service being advertised",
                      "aliases": ["product","product_name","sku","item","brand","category"]},
    "Impressions":   {"dtype": "int",   "required": True,  "description": "Number of ad impressions",
                      "aliases": ["impressions","impr","views","reach","exposures","ads_served","impr."]},
    "Clicks":        {"dtype": "int",   "required": True,  "description": "Number of ad clicks",
                      "aliases": ["clicks","click","link_clicks","ctr_raw","taps","link clicks"]},
    "Spend":         {"dtype": "float", "required": True,  "description": "Ad spend in currency units",
                      "aliases": ["spend","cost","budget_spent","amount_spent","expenditure","media_cost","investment"]},
    "Revenue":       {"dtype": "float", "required": False, "description": "Revenue generated",
                      "aliases": ["revenue","sales","income","return","gross_revenue","roas_value"]},
    "Conversions":   {"dtype": "int",   "required": False, "description": "Desired actions completed",
                      "aliases": ["conversions","conv","orders","leads","purchases","actions"]},
}

REQUIRED_COLUMNS = [k for k, v in TARGET_SCHEMA.items() if v["required"]]
ALL_COLUMNS      = list(TARGET_SCHEMA.keys())

print("Target schema columns:", ALL_COLUMNS)
print("Required:", REQUIRED_COLUMNS)


Target schema columns: ['Date', 'Campaign_Name', 'Channel', 'Region', 'Product', 'Impressions', 'Clicks', 'Spend', 'Revenue', 'Conversions']
Required: ['Date', 'Campaign_Name', 'Channel', 'Impressions', 'Clicks', 'Spend']


## Section 3 — Configure Google Gemini API

In [24]:
_GENAI_AVAILABLE = False
_gemini_client = None

try:
    from google import genai as _genai
    _GENAI_AVAILABLE = True
    if GEMINI_API_KEY:
        _gemini_client = _genai.Client(api_key=GEMINI_API_KEY)
        print(f"✅ Gemini client ready (model: gemini-2.5-flash)")
    else:
        print("⚠️  No API key — Gemini client not initialised. Using heuristic mapping only.")
except ImportError:
    print("⚠️  google-genai not installed. Using heuristic mapping only.")


def call_gemini(prompt: str, model: str = "gemini-2.5-flash") -> str:
    """Helper: call Gemini and return the response text (or empty string on failure)."""
    if not _gemini_client:
        return ""
    try:
        response = _gemini_client.models.generate_content(model=model, contents=prompt)
        return response.text.strip()
    except Exception as exc:
        print(f"Gemini call failed: {exc}")
        return ""

# Quick smoke-test (only if key present)
if _gemini_client:
    test_reply = call_gemini("Reply with exactly: GEMINI_OK")
    print("Gemini test response:", test_reply)


✅ Gemini client ready (model: gemini-2.5-flash)
Gemini test response: GEMINI_OK


## Section 4 — Data Ingestion (CSV / Excel)

In [25]:
def ingest_file(source, filename: str = "") -> pd.DataFrame:
    """Load CSV or Excel into a DataFrame."""
    fname = filename or (str(source) if isinstance(source, (str, Path)) else "file")
    ext   = Path(fname).suffix.lower()
    if ext in (".xls", ".xlsx", ".xlsm"):
        df = pd.read_excel(source, engine="openpyxl")
    elif ext == ".csv":
        df = pd.read_csv(source)
    else:
        try:
            df = pd.read_csv(source)
        except Exception:
            df = pd.read_excel(source, engine="openpyxl")
    print(f"✅ Loaded '{fname}': {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Columns: {list(df.columns)}")
    return df

# Quick demonstration with an in-memory CSV
_demo_csv = io.StringIO("""Date,Campaign_Name,Channel,Impressions,Clicks,Spend
2024-01-01,Summer Sale,Google Ads,50000,750,900.00
2024-01-02,Summer Sale,Facebook,30000,450,540.00
""")
df_demo = ingest_file(_demo_csv, filename="demo.csv")
df_demo.head()


✅ Loaded 'demo.csv': 2 rows × 6 columns
   Columns: ['Date', 'Campaign_Name', 'Channel', 'Impressions', 'Clicks', 'Spend']


,Date,Campaign_Name,Channel,Impressions,Clicks,Spend
0,2024-01-01,Summer Sale,Google Ads,50000,750,900.0
1,2024-01-02,Summer Sale,Facebook,30000,450,540.0


## Section 5 — Schema Recognition & Column Preview

In [26]:
def recognize_schema(df: pd.DataFrame) -> dict:
    """Compare df columns to TARGET_SCHEMA and classify each column."""
    cols = [c.strip() for c in df.columns]
    matched   = [c for c in cols if c in ALL_COLUMNS]
    extra     = [c for c in cols if c not in ALL_COLUMNS]
    missing   = [c for c in ALL_COLUMNS if c not in cols]
    missing_req = [c for c in REQUIRED_COLUMNS if c not in cols]

    print(f"✅ Matched            : {matched}")
    print(f"❓ Extra (not in schema): {extra}")
    print(f"⚠️  Missing from schema : {missing}")
    print(f"❌ Missing REQUIRED  : {missing_req}")

    # Summary table
    summary = pd.DataFrame({
        "Column":   cols,
        "In Schema": [c in ALL_COLUMNS for c in cols],
        "Required":  [TARGET_SCHEMA.get(c, {}).get("required", False) for c in cols],
        "Dtype":     [str(df[c].dtype) for c in cols],
        "Non-null":  [df[c].notna().sum() for c in cols],
        "Samples":   [", ".join(str(v) for v in df[c].dropna().head(3)) for c in cols],
    })
    return {"matched": matched, "extra": extra, "missing": missing,
            "missing_required": missing_req, "summary": summary}

info = recognize_schema(df_demo)
info["summary"]


✅ Matched            : ['Date', 'Campaign_Name', 'Channel', 'Impressions', 'Clicks', 'Spend']
❓ Extra (not in schema): []
⚠️  Missing from schema : ['Region', 'Product', 'Revenue', 'Conversions']
❌ Missing REQUIRED  : []


,Column,In Schema,Required,Dtype,Non-null,Samples
0,Date,True,True,object,2,"2024-01-01, 2024-01-02"
1,Campaign_Name,True,True,object,2,"Summer Sale, Summer Sale"
2,Channel,True,True,object,2,"Google Ads, Facebook"
3,Impressions,True,True,int64,2,"50000, 30000"
4,Clicks,True,True,int64,2,"750, 450"
5,Spend,True,True,float64,2,"900.0, 540.0"


## Section 6 — LLM-Assisted Column Mapping

In [27]:
CONFIDENCE_THRESHOLD = 0.70

def _normalize(name: str) -> str:
    return re.sub(r"[\s_\-\.]+", "_", name.strip().lower())

def heuristic_map(columns: list[str]) -> dict[str, dict]:
    """Alias-based mapping — fast, no LLM call needed.

    Confidence tiers:
      1.00 — Exact column-name match (column IS already a standard schema column)
      0.90 — Exact alias match (column normalised form found in alias list)
      0.65 — Partial / substring alias match
    """
    alias_index: dict[str, str] = {}
    for target, meta in TARGET_SCHEMA.items():
        for alias in meta["aliases"]:
            alias_index[_normalize(alias)] = target

    results = {}
    for col in columns:
        norm = _normalize(col)

        # ── Tier 1: exact column-name match ──────────────────────────────────
        if col in TARGET_SCHEMA:
            results[col] = {
                "target": col,
                "confidence": 1.0,
                "reasoning": f"Exact column-name match: '{col}' is already a standard column.",
            }
            continue

        # ── Tier 2: exact alias match ─────────────────────────────────────────
        if norm in alias_index:
            results[col] = {
                "target": alias_index[norm],
                "confidence": 0.90,
                "reasoning": f"Exact alias match: '{col}' → '{alias_index[norm]}'",
            }
        else:
            # ── Tier 3: partial / substring alias match ───────────────────────
            matched = [t for alias, t in alias_index.items() if alias in norm or norm in alias]
            if matched:
                results[col] = {
                    "target": matched[0],
                    "confidence": 0.65,
                    "reasoning": f"Partial alias match: '{col}' ≈ '{matched[0]}'",
                }
    return results


def llm_map(columns: list[str], sample_values: dict[str, list]) -> dict[str, dict]:
    """Ask Gemini to suggest mappings for given columns."""
    if not _gemini_client:
        return {}
    schema_text = "Target Schema columns and descriptions:\n" + "\n".join(
        f"  {k}: {v['description']}" for k, v in TARGET_SCHEMA.items()
    )
    cols_text = "\n".join(
        f"  - {c}: samples = {sample_values.get(c,[])[:3]}" for c in columns
    )
    prompt = f"""{schema_text}

Source columns to map:
{cols_text}

Return a JSON array ONLY (no markdown fences):
[{{"source_column":"...","target_column":"... or null","confidence":0.0-1.0,"reasoning":"..."}}]
"""
    raw = call_gemini(prompt)
    if not raw:
        return {}
    raw = re.sub(r"^```[a-z]*\n?", "", raw).rstrip("` \n")
    try:
        data = json.loads(raw)
        return {
            d["source_column"]: {
                "target": d.get("target_column"),
                "confidence": float(d.get("confidence", 0)),
                "reasoning": d.get("reasoning", ""),
            }
            for d in data
        }
    except Exception as exc:
        print(f"LLM parse error: {exc}\nRaw: {raw[:300]}")
        return {}


def suggest_full_mapping(df: pd.DataFrame) -> dict[str, dict]:
    """Combine heuristic + LLM mapping for all columns."""
    columns = list(df.columns)
    sample_values = {c: df[c].dropna().head(5).tolist() for c in columns}

    mappings = heuristic_map(columns)
    unresolved = [c for c in columns if c not in mappings or mappings[c]["confidence"] < CONFIDENCE_THRESHOLD]

    if unresolved:
        llm_results = llm_map(unresolved, sample_values)
        for col, m in llm_results.items():
            # Only accept LLM result if target is a valid schema column
            if m.get("target") is not None and m["target"] not in ALL_COLUMNS:
                print(f"  ⚠️  LLM returned invalid target '{m['target']}' for '{col}' — ignored.")
                continue
            if col not in mappings or m["confidence"] > mappings[col]["confidence"]:
                mappings[col] = m

    for col in columns:
        if col not in mappings:
            mappings[col] = {"target": None, "confidence": 0.0, "reasoning": "No match found"}

    print("\nMapping Suggestions:")
    for col, m in mappings.items():
        icon = "🟢" if m["confidence"] >= CONFIDENCE_THRESHOLD else "🔴"
        print(f"  {icon} {col:25s} → {str(m['target']):20s}  ({m['confidence']:.0%})  {m['reasoning']}")
    return mappings

# Demo: alias file
_alias_csv = io.StringIO("""Report Date,Campaign,Platform,Geo,Brand,Impr.,Link Clicks,Cost,Sales,Orders
2024-01-01,Alpha,Google Ads,US,Prod A,100000,1500,1800.00,9000.00,90
""")
df_alias = ingest_file(_alias_csv, filename="alias.csv")
mapping_suggestions = suggest_full_mapping(df_alias)


✅ Loaded 'alias.csv': 1 rows × 10 columns
   Columns: ['Report Date', 'Campaign', 'Platform', 'Geo', 'Brand', 'Impr.', 'Link Clicks', 'Cost', 'Sales', 'Orders']

Mapping Suggestions:
  🟢 Report Date               → Date                  (90%)  Exact alias match: 'Report Date' → 'Date'
  🟢 Campaign                  → Campaign_Name         (90%)  Exact alias match: 'Campaign' → 'Campaign_Name'
  🟢 Platform                  → Channel               (90%)  Exact alias match: 'Platform' → 'Channel'
  🟢 Geo                       → Region                (90%)  Exact alias match: 'Geo' → 'Region'
  🟢 Brand                     → Product               (90%)  Exact alias match: 'Brand' → 'Product'
  🟢 Impr.                     → Impressions           (90%)  Exact alias match: 'Impr.' → 'Impressions'
  🟢 Link Clicks               → Clicks                (90%)  Exact alias match: 'Link Clicks' → 'Clicks'
  🟢 Cost                      → Spend                 (90%)  Exact alias match: 'Cost' → 'Spend'

## Section 7 — Human-in-the-Loop Mapping Confirmation (Notebook Mode)

In [28]:
# Operation log — accumulated throughout the notebook
LOGS: list[dict] = []

def log(operation: str, detail: str, status: str = "ok"):
    LOGS.append({"timestamp": datetime.now().isoformat(timespec="seconds"),
                 "operation": operation, "detail": detail, "status": status})


def confirm_mappings_notebook(suggestions: dict[str, dict]) -> dict[str, str | None]:
    """
    In notebook mode: auto-accept high-confidence, prompt CLI for low-confidence.
    In the full app, the Streamlit UI handles this interactively.
    """
    confirmed: dict[str, str | None] = {}
    for col, m in suggestions.items():
        if m["confidence"] >= CONFIDENCE_THRESHOLD:
            confirmed[col] = m["target"]
            log("auto_accept", f"'{col}' → '{m['target']}' ({m['confidence']:.0%})", "ok")
        else:
            # In notebook: prompt user
            print(f"\n🔴 Low confidence for column '{col}'")
            print(f"   Suggestion : {m['target']}  ({m['confidence']:.0%})")
            print(f"   Reasoning  : {m['reasoning']}")
            print(f"   Options    : {ALL_COLUMNS}")
            choice = input(f"   Accept suggestion '{m['target']}'? [ENTER] or type target / 'drop': ").strip()
            if choice == "" and m["target"]:
                confirmed[col] = m["target"]
                log("user_confirmed", f"'{col}' → '{m['target']}'", "user_input")
            elif choice.lower() == "drop" or choice == "":
                confirmed[col] = None
                log("user_dropped", f"'{col}' dropped by user", "user_input")
            elif choice in ALL_COLUMNS:
                confirmed[col] = choice
                log("user_override", f"'{col}' → '{choice}' (overridden)", "user_input")
            else:
                confirmed[col] = None
                log("user_dropped", f"'{col}' unrecognised input → dropped", "warning")
    return confirmed

# In non-interactive mode (CI / testing), auto-confirm everything
def auto_confirm(suggestions: dict[str, dict]) -> dict[str, str | None]:
    confirmed = {}
    for col, m in suggestions.items():
        confirmed[col] = m["target"]
        log("auto_accept", f"'{col}' → '{m['target']}'", "ok")
    return confirmed

confirmed_mappings = auto_confirm(mapping_suggestions)
print("\nConfirmed mappings:", confirmed_mappings)



Confirmed mappings: {'Report Date': 'Date', 'Campaign': 'Campaign_Name', 'Platform': 'Channel', 'Geo': 'Region', 'Brand': 'Product', 'Impr.': 'Impressions', 'Link Clicks': 'Clicks', 'Cost': 'Spend', 'Sales': 'Revenue', 'Orders': 'Conversions'}


## Section 8 — Data Transformation: Unpivoting (Wide → Long)

In [29]:
def _col_str(c) -> str:
    """Normalise a column label to a clean string (handles Timestamp objects)."""
    if hasattr(c, "strftime"):
        return c.strftime("%d/%m/%Y")
    s = str(c)
    # Strip trailing time components: "2023-01-07 00:00:00" → "2023-01-07"
    s = re.sub(r"\s+\d{2}:\d{2}:\d{2}$", "", s).strip()
    return s


def detect_wide_format(df: pd.DataFrame) -> dict:
    """Detect pivoted (wide) format DataFrame.

    Handles both string date headers and Timestamp objects that Excel
    sometimes produces when column headers look like dates.
    """
    # Build mapping: normalised_str → original column object
    col_map = {_col_str(c): c for c in df.columns}
    col_names = list(col_map.keys())

    # Pattern A: period names (Jan-2024, Q1-2024, …)
    period_re = re.compile(
        r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec"
        r"|january|february|march|april|june|july|august"
        r"|september|october|november|december"
        r"|q[1-4]|20\d{2}[-_]?\d{0,2}|\d{4}[-_/]\d{2})",
        re.IGNORECASE,
    )
    # Pattern B: date strings (dd/mm/yyyy, yyyy-mm-dd)
    date_col_re = re.compile(
        r"^(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}|\d{4}[/\-]\d{2}[/\-]\d{2})$",
        re.IGNORECASE,
    )

    period_vars = [c for c in col_names if period_re.match(c)]
    date_vars   = [c for c in col_names if date_col_re.match(c)]
    value_vars_names = date_vars if len(date_vars) >= 2 else period_vars

    if len(value_vars_names) >= 2:
        id_var_names = [c for c in col_names if c not in value_vars_names]
        id_vars    = [col_map[c] for c in id_var_names]
        value_vars = [col_map[c] for c in value_vars_names]
        wide_type  = "date_string" if date_vars else "period_name"
        print(f"⚠️  Wide format detected: {len(value_vars)} period cols →"
              f" {[_col_str(v) for v in value_vars[:5]]}…")
        return {"is_wide": True, "id_vars": id_vars, "value_vars": value_vars,
                "wide_type": wide_type}
    print("✅ Not wide format.")
    return {"is_wide": False, "id_vars": [], "value_vars": [], "wide_type": None}


def apply_melt(df: pd.DataFrame, id_vars: list, value_vars: list,
               var_name: str = "Date", value_name: str = "Spends") -> pd.DataFrame:
    df_long = pd.melt(df, id_vars=id_vars, value_vars=value_vars,
                      var_name=var_name, value_name=value_name)

    # Normalise the var column in case headers were Timestamps
    if var_name in df_long.columns and len(df_long) > 0:
        df_long[var_name] = df_long[var_name].apply(
            lambda x: x.strftime("%d/%m/%Y") if hasattr(x, "strftime") else str(x)
        )
        # Also strip trailing timestamps: "2023-01-07 00:00:00" → "07/01/2023"
        sample = df_long[var_name].iloc[0]
        if re.match(r"\d{4}-\d{2}-\d{2}", str(sample)):
            parsed = pd.to_datetime(df_long[var_name], errors="coerce")
            df_long[var_name] = parsed.dt.strftime("%d/%m/%Y").where(
                parsed.notna(), df_long[var_name]
            )

    log("melt", f"Melted {len(value_vars)} cols → long format ({len(df_long)} rows)", "ok")
    print(f"✅ Melted → {df_long.shape[0]:,} rows × {df_long.shape[1]} cols")
    return df_long


# Demo — load the updated wide format (now has Impressions as id-var)
_wide_csv = io.StringIO(
    "Category,Sub_Category,Channel,Publisher,Impressions,Jan-2024,Feb-2024,Mar-2024\n"
    "Haircare,Shampoo,TV,Sony LIV,1500000,5000,6200,4800\n"
    "Beverages,Cola,Digital,Google,800000,3000,3500,3200\n"
)
df_wide = ingest_file(_wide_csv, filename="wide.csv")
wide_info = detect_wide_format(df_wide)
if wide_info["is_wide"]:
    df_long = apply_melt(df_wide, wide_info["id_vars"], wide_info["value_vars"])
    print(df_long)


✅ Loaded 'wide.csv': 2 rows × 8 columns
   Columns: ['Category', 'Sub_Category', 'Channel', 'Publisher', 'Impressions', 'Jan-2024', 'Feb-2024', 'Mar-2024']
⚠️  Wide format detected: 3 period cols → ['Jan-2024', 'Feb-2024', 'Mar-2024']…
✅ Melted → 6 rows × 7 cols
    Category Sub_Category  Channel Publisher  Impressions      Date  Spends
0   Haircare      Shampoo       TV  Sony LIV      1500000  Jan-2024    5000
1  Beverages         Cola  Digital    Google       800000  Jan-2024    3000
2   Haircare      Shampoo       TV  Sony LIV      1500000  Feb-2024    6200
3  Beverages         Cola  Digital    Google       800000  Feb-2024    3500
4   Haircare      Shampoo       TV  Sony LIV      1500000  Mar-2024    4800
5  Beverages         Cola  Digital    Google       800000  Mar-2024    3200


## Section 8b — Schema-Wide Format Detection (Tool 2b)

Detects when **categorical field values** (e.g. channel names like *TV*, *Digital*, *Radio*) are used as column headers instead of a canonical `Channel` column.  
The heuristic checks whether ≥ 2 column headers match known values for Channel, Publisher, or Category.  
A fallback LLM prompt is used when the heuristic is inconclusive.

In [30]:
# ── Known categorical values used in heuristic detection ──────────────────────
_KNOWN_CHANNEL_VALUES    = {"tv", "digital", "radio", "ooh", "print", "cinema",
                             "social", "search", "display"}
_KNOWN_PUBLISHER_VALUES  = {"sony", "google", "star", "zee", "hotstar", "youtube",
                             "facebook", "instagram", "twitter"}
_KNOWN_CATEGORY_VALUES   = {"haircare", "beverages", "food", "skincare", "dairy",
                             "homecare", "personal care", "snacks"}

_FIELD_KNOWN_VALUES = {
    "Channel":   _KNOWN_CHANNEL_VALUES,
    "Publisher": _KNOWN_PUBLISHER_VALUES,
    "Category":  _KNOWN_CATEGORY_VALUES,
}


def detect_schema_wide_format(df: pd.DataFrame) -> dict:
    """
    Tool 2b — Detect schema-wide format: categorical field values used as column headers.

    Example input (schema-wide — Channel values are column headers):
        Category | Sub_Category | TV    | Digital | Radio
        Haircare | Shampoo      | 5000  | 3000    | 1200

    Expected output:
        Category | Sub_Category | Channel  | Spends
        Haircare | Shampoo      | TV       | 5000
        Haircare | Shampoo      | Digital  | 3000
        ...

    Returns a dict with keys:
        is_schema_wide   (bool)
        schema_field     (str | None)  – e.g. "Channel"
        id_vars          (list)
        value_vars       (list)
        suggested_value_col (str)      – name for the melted value column
        detection_method (str)
    """
    col_lower = {c.lower().strip(): c for c in df.columns}

    for field, known in _FIELD_KNOWN_VALUES.items():
        matches = [orig for norm, orig in col_lower.items() if norm in known]
        if len(matches) >= 2:
            id_vars    = [c for c in df.columns if c not in matches]
            value_vars = matches
            print(f"⚠️  Schema-wide format detected: '{field}' values are column headers "
                  f"({[str(m) for m in matches]})")
            log("detect_schema_wide", f"{field} values used as headers: {matches}", "ok")
            return {
                "is_schema_wide":      True,
                "schema_field":        field,
                "id_vars":             id_vars,
                "value_vars":          value_vars,
                "suggested_value_col": "Spends",
                "detection_method":    "heuristic",
            }

    print("✅ No schema-wide format detected.")
    log("detect_schema_wide", "Not schema-wide", "ok")
    return {
        "is_schema_wide": False, "schema_field": None,
        "id_vars": [], "value_vars": [], "suggested_value_col": "Spends",
        "detection_method": "heuristic",
    }


def apply_schema_wide_melt(df: pd.DataFrame, schema_info: dict) -> pd.DataFrame:
    """Melt a schema-wide DataFrame into long format."""
    field      = schema_info["schema_field"]
    id_vars    = schema_info["id_vars"]
    value_vars = schema_info["value_vars"]
    val_col    = schema_info.get("suggested_value_col", "Spends")

    df_long = pd.melt(df, id_vars=id_vars, value_vars=value_vars,
                      var_name=field, value_name=val_col)
    log("schema_wide_melt", f"Melted {len(value_vars)} schema cols → {len(df_long)} rows", "ok")
    print(f"✅ Schema-wide melt → {df_long.shape[0]:,} rows × {df_long.shape[1]} cols")
    return df_long


# ── Demo ──────────────────────────────────────────────────────────────────────
_schema_wide_csv = io.StringIO(
    "Category,Sub_Category,TV,Digital,Radio\n"
    "Haircare,Shampoo,5000,3000,1200\n"
    "Beverages,Cola,8000,6200,2100\n"
)
df_schema_wide = ingest_file(_schema_wide_csv, filename="schema_wide.csv")
print("Input DataFrame:")
print(df_schema_wide.to_string(index=False))

sw_info = detect_schema_wide_format(df_schema_wide)
if sw_info["is_schema_wide"]:
    df_schema_long = apply_schema_wide_melt(df_schema_wide, sw_info)
    print("\nOutput (long format):")
    print(df_schema_long.to_string(index=False))


✅ Loaded 'schema_wide.csv': 2 rows × 5 columns
   Columns: ['Category', 'Sub_Category', 'TV', 'Digital', 'Radio']
Input DataFrame:
 Category Sub_Category   TV  Digital  Radio
 Haircare      Shampoo 5000     3000   1200
Beverages         Cola 8000     6200   2100
⚠️  Schema-wide format detected: 'Channel' values are column headers (['TV', 'Digital', 'Radio'])
✅ Schema-wide melt → 6 rows × 4 cols

Output (long format):
 Category Sub_Category Channel  Spends
 Haircare      Shampoo      TV    5000
Beverages         Cola      TV    8000
 Haircare      Shampoo Digital    3000
Beverages         Cola Digital    6200
 Haircare      Shampoo   Radio    1200
Beverages         Cola   Radio    2100


## Section 9 — Data Transformation: Date Parts Combination

In [31]:
def detect_split_date(df: pd.DataFrame) -> dict:
    """Check if separate day/month/year columns are present."""
    col_lower = {c.lower(): c for c in df.columns}
    day   = next((col_lower[k] for k in col_lower if re.search(r"\bday\b",   k)), None)
    month = next((col_lower[k] for k in col_lower if re.search(r"\bmonth\b", k)), None)
    year  = next((col_lower[k] for k in col_lower if re.search(r"\byear\b",  k)), None)
    found = bool(day and month and year)
    if found:
        print(f"📅 Split date detected: year='{year}', month='{month}', day='{day}'")
    else:
        print("✅ No split date columns detected.")
    return {"has_split_date": found, "day_col": day, "month_col": month, "year_col": year}


def combine_split_date(df: pd.DataFrame, year_col: str, month_col: str,
                       day_col: str, out_col: str = "Date") -> pd.DataFrame:
    """Combine year/month/day columns into a single datetime column."""
    df = df.copy()
    df[out_col] = pd.to_datetime(
        dict(year=df[year_col].astype(int),
             month=df[month_col].astype(int),
             day=df[day_col].astype(int)),
        errors="coerce",
    )
    df = df.drop(columns=[year_col, month_col, day_col], errors="ignore")
    log("combine_date", f"Combined {year_col}/{month_col}/{day_col} → '{out_col}'", "ok")
    print(f"✅ Created '{out_col}': {df[out_col].head(3).tolist()}")
    return df


# Demo
_split_csv = io.StringIO(
    "day,month,year,Campaign_Name,Channel,Impressions,Clicks,Spend\n"
    "1,3,2024,Alpha,Google Ads,10000,200,500.00\n"
    "15,3,2024,Beta,Facebook,20000,400,1000.00\n"
)
df_split = ingest_file(_split_csv, filename="split.csv")
date_info = detect_split_date(df_split)
if date_info["has_split_date"]:
    df_split = combine_split_date(df_split, date_info["year_col"], date_info["month_col"], date_info["day_col"])
print(df_split)


✅ Loaded 'split.csv': 2 rows × 8 columns
   Columns: ['day', 'month', 'year', 'Campaign_Name', 'Channel', 'Impressions', 'Clicks', 'Spend']
📅 Split date detected: year='year', month='month', day='day'
✅ Created 'Date': [Timestamp('2024-03-01 00:00:00'), Timestamp('2024-03-15 00:00:00')]
  Campaign_Name     Channel  Impressions  Clicks   Spend       Date
0         Alpha  Google Ads        10000     200   500.0 2024-03-01
1          Beta    Facebook        20000     400  1000.0 2024-03-15


## Section 9b — Metric Hierarchy Unstack (Tool 3b)

Detects columns whose names encode **both a metric and a categorical value**, e.g. `TV_Spends`, `Digital_Impressions`.  
The heuristic looks for `{value}_{metric}` or `{metric}_{value}` patterns using known schema metrics (`Spends`, `Impressions`) and known categorical values (Channel, Publisher, Period).  
The detected columns are melted into a long-format DataFrame with separate `split_field` and `metric_col` columns.

In [32]:
_KNOWN_METRICS = {"spends", "impressions", "clicks", "revenue", "conversions", "grp"}

_KNOWN_SPLIT_VALUES = (
    _KNOWN_CHANNEL_VALUES  |       # TV, Digital, Radio, OOH …
    _KNOWN_PUBLISHER_VALUES |      # Sony, Google, Star …
    {f"q{i}" for i in range(1, 5)} |   # Q1–Q4
    {"jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"}
)


def detect_metric_hierarchy(df: pd.DataFrame) -> dict:
    """
    Tool 3b — Detect metric-hierarchy columns such as TV_Spends, Digital_Impressions.

    Pattern: <split_value>_<metric>  OR  <metric>_<split_value>  (case-insensitive)

    Returns:
        has_metric_hierarchy (bool)
        metric_col   (str)  – the metric being split (e.g. "Spends")
        split_field  (str)  – the schema field the split represents (e.g. "Channel")
        split_values (list) – categorical values extracted (e.g. ["TV", "Digital"])
        matched_columns (list)
        id_vars      (list)
    """
    matched = {}   # col_name → {"split_value": ..., "metric": ...}

    for col in df.columns:
        parts = re.split(r"[_\s\-]", col.strip(), maxsplit=1)
        if len(parts) < 2:
            continue
        a, b = parts[0].lower(), parts[1].lower()
        if a in _KNOWN_SPLIT_VALUES and b in _KNOWN_METRICS:
            matched[col] = {"split_value": parts[0], "metric": parts[1].capitalize()}
        elif a in _KNOWN_METRICS and b in _KNOWN_SPLIT_VALUES:
            matched[col] = {"split_value": parts[1], "metric": parts[0].capitalize()}

    if len(matched) >= 2:
        metrics = list({v["metric"] for v in matched.values()})
        splits  = list({v["split_value"] for v in matched.values()})
        metric_col = metrics[0]

        # Determine which schema field the split values belong to
        split_lower = {s.lower() for s in splits}
        split_field = "Channel"  # default
        for field, known in _FIELD_KNOWN_VALUES.items():
            if split_lower & known:
                split_field = field
                break

        id_vars = [c for c in df.columns if c not in matched]
        print(f"⚠️  Metric hierarchy detected: metric='{metric_col}', "
              f"field='{split_field}', values={splits}")
        log("detect_metric_hierarchy",
            f"metric={metric_col}, field={split_field}, cols={list(matched.keys())}", "ok")
        return {
            "has_metric_hierarchy": True,
            "metric_col":    metric_col,
            "split_field":   split_field,
            "split_values":  splits,
            "matched_columns": list(matched.keys()),
            "id_vars":       id_vars,
        }

    print("✅ No metric hierarchy detected.")
    log("detect_metric_hierarchy", "None found", "ok")
    return {
        "has_metric_hierarchy": False, "metric_col": None,
        "split_field": None, "split_values": [], "matched_columns": [], "id_vars": [],
    }


def apply_metric_hierarchy_unstack(df: pd.DataFrame, hier_info: dict) -> pd.DataFrame:
    """Melt metric-hierarchy columns into long format."""
    matched_cols = hier_info["matched_columns"]
    id_vars      = hier_info["id_vars"]
    split_field  = hier_info["split_field"]
    metric_col   = hier_info["metric_col"]

    df_long = pd.melt(df, id_vars=id_vars, value_vars=matched_cols,
                      var_name=split_field, value_name=metric_col)

    # Strip the metric suffix/prefix to leave only the categorical label
    metric_lower = metric_col.lower()
    df_long[split_field] = df_long[split_field].str.replace(
        rf"[_\-\s]?{re.escape(metric_lower)}$", "", flags=re.IGNORECASE, regex=True
    ).str.replace(
        rf"^{re.escape(metric_lower)}[_\-\s]?", "", flags=re.IGNORECASE, regex=True
    ).str.title()

    log("metric_hierarchy_unstack",
        f"Melted {len(matched_cols)} metric cols → {len(df_long)} rows", "ok")
    print(f"✅ Metric hierarchy unstack → {df_long.shape[0]:,} rows × {df_long.shape[1]} cols")
    return df_long


# ── Demo ──────────────────────────────────────────────────────────────────────
_metric_hier_csv = io.StringIO(
    "Category,Sub_Category,Date,TV_Spends,Digital_Spends,Radio_Spends\n"
    "Haircare,Shampoo,01/01/2024,5000,3000,1200\n"
    "Beverages,Cola,01/01/2024,8000,6200,2100\n"
)
df_metric = ingest_file(_metric_hier_csv, filename="metric_hier.csv")
print("Input DataFrame:")
print(df_metric.to_string(index=False))

mh_info = detect_metric_hierarchy(df_metric)
if mh_info["has_metric_hierarchy"]:
    df_metric_long = apply_metric_hierarchy_unstack(df_metric, mh_info)
    print("\nOutput (long format):")
    print(df_metric_long.to_string(index=False))


✅ Loaded 'metric_hier.csv': 2 rows × 6 columns
   Columns: ['Category', 'Sub_Category', 'Date', 'TV_Spends', 'Digital_Spends', 'Radio_Spends']
Input DataFrame:
 Category Sub_Category       Date  TV_Spends  Digital_Spends  Radio_Spends
 Haircare      Shampoo 01/01/2024       5000            3000          1200
Beverages         Cola 01/01/2024       8000            6200          2100
⚠️  Metric hierarchy detected: metric='Spends', field='Channel', values=['Radio', 'TV', 'Digital']
✅ Metric hierarchy unstack → 6 rows × 5 cols

Output (long format):
 Category Sub_Category       Date Channel  Spends
 Haircare      Shampoo 01/01/2024      Tv    5000
Beverages         Cola 01/01/2024      Tv    8000
 Haircare      Shampoo 01/01/2024 Digital    3000
Beverages         Cola 01/01/2024 Digital    6200
 Haircare      Shampoo 01/01/2024   Radio    1200
Beverages         Cola 01/01/2024   Radio    2100


## Section 10 — Type Validation with Pydantic

In [33]:
from typing import Optional
from pydantic import BaseModel, field_validator, ValidationError

class SchemaRow(BaseModel):
    """Pydantic model for one output row. Optional fields may be None."""
    Date:          Optional[str]   = None
    Campaign_Name: Optional[str]   = None
    Channel:       Optional[str]   = None
    Region:        Optional[str]   = None
    Product:       Optional[str]   = None
    Impressions:   Optional[int]   = None
    Clicks:        Optional[int]   = None
    Spend:         Optional[float] = None
    Revenue:       Optional[float] = None
    Conversions:   Optional[int]   = None

    @field_validator("Impressions", "Clicks", "Conversions", mode="before")
    @classmethod
    def coerce_int(cls, v):
        try: return int(float(v)) if v is not None else None
        except: return None

    @field_validator("Spend", "Revenue", mode="before")
    @classmethod
    def coerce_float(cls, v):
        try: return float(v) if v is not None else None
        except: return None


def _coerce_date_series(series: pd.Series) -> pd.Series:
    """
    Robustly coerce a Series of date strings to 'DD/MM/YYYY' format.

    Handles:
    - Plain date strings: '07/01/2023'
    - Hierarchical unstack output: '07/01/2023, Saturday'
    - ISO format: '2023-01-07'
    - Timestamps (datetime objects)
    """
    raw = series.astype(str)

    # Extract only the date portion (strips weekday name if present)
    cleaned = raw.str.extract(
        r"^(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}|\d{4}[/\-]\d{2}[/\-]\d{2})",
        expand=False,
    ).fillna(raw)

    # Parse with dayfirst=True (DD/MM/YYYY convention)
    parsed = pd.to_datetime(cleaned, dayfirst=True, errors="coerce")

    # Fallback: try ISO format for any still-NaT rows
    failed = parsed.isna() & series.notna()
    if failed.any():
        parsed[failed] = pd.to_datetime(raw[failed], errors="coerce")

    # Return as formatted strings; preserve original value if parsing failed
    return parsed.dt.strftime("%d/%m/%Y").where(parsed.notna(), series.astype(str))


def validate_and_coerce(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:
    """Validate each row against SchemaRow; coerce types and collect errors."""
    valid_rows, errors = [], []
    known_cols = list(SchemaRow.model_fields.keys())
    available  = [c for c in known_cols if c in df.columns]

    # Pre-coerce the Date column before row-level validation
    df = df.copy()
    if "Date" in df.columns:
        df["Date"] = _coerce_date_series(df["Date"])

    for idx, row in df.iterrows():
        row_dict = {c: (None if pd.isna(row[c]) else row[c]) for c in available}
        try:
            parsed = SchemaRow(**row_dict)
            valid_rows.append(parsed.model_dump())
        except ValidationError as e:
            errors.append({"row": idx, "errors": e.errors()})
            valid_rows.append(row_dict)

    df_out = pd.DataFrame(valid_rows)
    # Restore non-schema extra columns
    for col in df.columns:
        if col not in df_out.columns:
            df_out[col] = df[col].values
    if errors:
        print(f"⚠️  Validation errors in {len(errors)} row(s).")
        log("validate", f"{len(errors)} validation errors", "warning")
    else:
        print("✅ All rows validated successfully.")
        log("validate", "All rows passed validation", "ok")
    return df_out, errors

# Demo
df_validated, ve = validate_and_coerce(df_demo)
print(df_validated.dtypes)
print("\nSample dates:", df_validated["Date"].head(3).tolist())


✅ All rows validated successfully.
Date              object
Campaign_Name     object
Channel           object
Region            object
Product           object
Impressions        int64
Clicks             int64
Spend            float64
Revenue           object
Conversions       object
dtype: object

Sample dates: ['01/01/2024', '01/02/2024']


## Section 11 — Apply Final Mappings & Transformations

In [34]:
def apply_confirmed_mappings(df: pd.DataFrame, confirmed: dict[str, str | None]) -> pd.DataFrame:
    """Rename columns per confirmed mapping, drop unmapped columns."""
    df = df.copy()
    rename_dict = {src: tgt for src, tgt in confirmed.items() if tgt is not None}
    drop_cols   = [src for src, tgt in confirmed.items() if tgt is None and src in df.columns]

    df = df.rename(columns=rename_dict)
    df = df.drop(columns=drop_cols, errors="ignore")

    for src, tgt in rename_dict.items():
        log("rename", f"'{src}' → '{tgt}'", "ok")
    for col in drop_cols:
        log("drop", f"Dropped '{col}'", "warning")
    print(f"✅ Renamed {len(rename_dict)} cols, dropped {len(drop_cols)} cols.")
    print(f"   Output columns: {list(df.columns)}")
    return df


# Full pipeline demo on alias file
df_work = df_alias.copy()
wi = detect_wide_format(df_work)
di = detect_split_date(df_work)
if di["has_split_date"]:
    df_work = combine_split_date(df_work, di["year_col"], di["month_col"], di["day_col"])
if wi["is_wide"]:
    df_work = apply_melt(df_work, wi["id_vars"], wi["value_vars"])
df_work = apply_confirmed_mappings(df_work, confirmed_mappings)
df_work, _ = validate_and_coerce(df_work)
print(df_work)


✅ Not wide format.
✅ No split date columns detected.
✅ Renamed 10 cols, dropped 0 cols.
   Output columns: ['Date', 'Campaign_Name', 'Channel', 'Region', 'Product', 'Impressions', 'Clicks', 'Spend', 'Revenue', 'Conversions']
✅ All rows validated successfully.
         Date Campaign_Name     Channel Region Product  Impressions  Clicks  \
0  01/01/2024         Alpha  Google Ads     US  Prod A       100000    1500   

    Spend  Revenue  Conversions  
0  1800.0   9000.0           90  


## Section 12 — Logging System

In [35]:
def logs_as_dataframe() -> pd.DataFrame:
    return pd.DataFrame(LOGS) if LOGS else pd.DataFrame(
        columns=["timestamp", "operation", "detail", "status"])

print(f"Log entries so far: {len(LOGS)}")
df_logs = logs_as_dataframe()
df_logs


Log entries so far: 28


,timestamp,operation,detail,status
0,2026-03-01T20:43:54,auto_accept,'Report Date' → 'Date',ok
1,2026-03-01T20:43:54,auto_accept,'Campaign' → 'Campaign_Name',ok
2,2026-03-01T20:43:54,auto_accept,'Platform' → 'Channel',ok
3,2026-03-01T20:43:54,auto_accept,'Geo' → 'Region',ok
4,2026-03-01T20:43:54,auto_accept,'Brand' → 'Product',ok
5,2026-03-01T20:43:54,auto_accept,'Impr.' → 'Impressions',ok
6,2026-03-01T20:43:54,auto_accept,'Link Clicks' → 'Clicks',ok
7,2026-03-01T20:43:54,auto_accept,'Cost' → 'Spend',ok
8,2026-03-01T20:43:54,auto_accept,'Sales' → 'Revenue',ok
9,2026-03-01T20:43:54,auto_accept,'Orders' → 'Conversions',ok


## Section 13 — Export Results & Logs to Excel

In [36]:
def export_to_excel(df_processed: pd.DataFrame, output_path: str | Path = "output/mapped_output.xlsx") -> Path:
    """Write processed data + logs to a multi-sheet Excel workbook."""
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(out, engine="xlsxwriter") as writer:
        df_processed.to_excel(writer, sheet_name="Results", index=False)
        logs_as_dataframe().to_excel(writer, sheet_name="Logs", index=False)

        # Auto-fit column widths
        for sheet_name, df_sheet in [("Results", df_processed), ("Logs", logs_as_dataframe())]:
            ws = writer.sheets[sheet_name]
            for i, col in enumerate(df_sheet.columns):
                max_w = max(len(str(col)),
                            df_sheet[col].astype(str).str.len().max() if len(df_sheet) else 0)
                ws.set_column(i, i, min(max_w + 2, 50))

    log("export", f"Saved → '{out}'", "ok")
    print(f"✅ Excel saved: {out}")
    return out


# Export the alias demo
out_path = export_to_excel(df_work, output_path="output/demo_mapped_output.xlsx")
print(f"📁 File: {out_path.resolve()}")

# Verify sheets
xl = pd.ExcelFile(out_path)
print("Sheets:", xl.sheet_names)


✅ Excel saved: output\demo_mapped_output.xlsx
📁 File: C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\output\demo_mapped_output.xlsx
Sheets: ['Results', 'Logs']


## Section 14 — Generate Sample Test Excel/CSV Files

In [37]:
import subprocess, sys

# Run the dedicated generator script
result = subprocess.run(
    [sys.executable, "generate_test_data.py"],
    capture_output=True, text=True, cwd=str(Path().resolve()),
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

# List generated files
data_dir = Path("data")
if data_dir.exists():
    for f in sorted(data_dir.iterdir()):
        size = f.stat().st_size
        print(f"  {f.name:35s}  {size:>8,} bytes")


Generating Kantar Media sample datasets...

[OK] C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\data\media_standard.csv  (90 rows)  â€” raw media dataset
[OK] C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\data\media_alias_names.xlsx  (90 rows)  â€” Tool 1 (Rename) demo
[OK] C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\data\media_wide_format.xlsx  (54 id-rows Ã— 12 date-columns)  â€” Tool 2 (Un-pivot) demo
[OK] C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\data\media_hierarchical_date.xlsx  (90 rows)  â€” Tool 3 (Hierarchical Unstack) demo

âœ…  All datasets written to data/
    media_standard.csv            <- raw 90-row media dataset
    media_alias_names.xlsx        <- Tool 1 (Rename) demo
    media_wide_format.xlsx        <- Tool 2 (Un-pivot) demo
    media_hierarchical_date.xlsx  <- Tool 3 (Hierarchical Unstack) demo


  media_alias_names.xlsx                 10,050 bytes
  media_hierarchical_date.x

## Section 15 — Run Unit Tests

In [38]:
import pytest

# Run tests from the tests/ directory and capture output
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True, cwd=str(Path().resolve()),
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode == 0:
    print("\n🎉 ALL TESTS PASSED!")
else:
    print("\n⚠️  Some tests failed. See output above.")
    if result.stderr:
        print(result.stderr[:1000])


============================= test session starts =============================
collecting ... collected 29 items

tests/test_agent.py::TestIngestion::test_ingest_csv PASSED               [  3%]
tests/test_agent.py::TestIngestion::test_ingest_excel PASSED             [  6%]
tests/test_agent.py::TestIngestion::test_ingest_bytesio_csv PASSED       [ 10%]
tests/test_agent.py::TestIngestion::test_ingest_bad_file_raises PASSED   [ 13%]
tests/test_agent.py::TestHeuristicMapping::test_exact_alias_match PASSED [ 17%]
tests/test_agent.py::TestHeuristicMapping::test_standard_columns_high_confidence PASSED [ 20%]
tests/test_agent.py::TestHeuristicMapping::test_unmapped_column_gets_none PASSED [ 24%]
tests/test_agent.py::TestTransformations::test_combine_split_date PASSED [ 27%]
tests/test_agent.py::TestTransformations::test_melt_wide_format PASSED   [ 31%]
tests/test_agent.py::TestTransformations::test_detect_wide_format PASSED [ 34%]
tests/test_agent.py::TestTransformations::test_detect_split_da

## Section 16 — Streamlit UI Application

The full Streamlit app lives in `agent/app.py`.  
Run the cell below to **launch the UI** in your browser.  
> **Tip:** Set your `GEMINI_API_KEY` in the sidebar of the app, or as an environment variable before launching.

In [39]:
import subprocess, sys
from pathlib import Path

app_path = Path("agent") / "app.py"
print(f"Launching Streamlit app: {app_path.resolve()}")
print("➡️  Open your browser at: http://localhost:8501")
print("   (Press Ctrl+C or interrupt the kernel to stop)\n")

# Launch as background process from the notebook
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(app_path),
     "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=str(Path().resolve()),
)
print(f"✅ Streamlit started (PID={proc.pid}). Navigate to http://localhost:8501")


Launching Streamlit app: C:\Users\HAMZU\OneDrive - iTouch Solutions\Desktop\Agentic AI Task\agent\app.py
➡️  Open your browser at: http://localhost:8501
   (Press Ctrl+C or interrupt the kernel to stop)

✅ Streamlit started (PID=22924). Navigate to http://localhost:8501
